In [2]:
import pandas as pd
import os
import json
import sys

try:
    current_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    current_dir = os.getcwd()

project_root = os.path.abspath(os.path.join(current_dir, '..'))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.preprocess import TextPreprocess

input_file = os.path.join(project_root, 'data', 'processed.csv')
output_file = os.path.join(project_root, 'data', 'processed_v2.csv')
edge_cases_file = os.path.join(project_root, 'tests', 'edge_cases.jsonl') 

def main():
    if not os.path.exists(input_file):
        print(f"Помилка: Файл {input_file} не знайдено!")
        print(f"Поточна директорія: {os.getcwd()}")
        return

    tp = TextPreprocess()
    df = pd.read_csv(input_file)
    text_col = df.columns[0] 

    print(f"--- ЗАПУСК ПАЙПЛАЙНУ ЛР2 ---")

    full_results = df[text_col].apply(lambda x: tp.preprocess(str(x)))
    df['text_v2'] = full_results.apply(lambda x: x['final'])

    char_counts = df['text_v2'].str.len()
    word_counts = df['text_v2'].str.split().str.len()

    avg_chars = char_counts.mean()
    med_chars = char_counts.median()
    avg_words = word_counts.mean()
    med_words = word_counts.median()

    duplicates = df.duplicated(subset=['text_v2']).sum()
    short_texts = (word_counts < 5).sum()
    garbage = df['text_v2'].apply(lambda x: not any(c.isalpha() for c in x)).sum()

    print(f"\n--- РЕЗУЛЬТАТИ АУДИТУ (ПІСЛЯ ЛР2) ---")
    print(f"1. Точні дублікати: {duplicates} ({duplicates/len(df)*100:.1f}%)")
    print(f"2. Короткі тексти (<5 слів): {short_texts}")
    print(f"3. Сміттєві рядки (без літер): {garbage}")
    print(f"4. Середня довжина: {avg_words:.1f} слів ({avg_chars:.1f} симв.)")
    print(f"5. Медіанна довжина: {med_words:.0f} слів ({med_chars:.0f} симв.)")

    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    df[['text_v2']].to_csv(output_file, index=False, encoding='utf-8-sig')

    os.makedirs(os.path.dirname(edge_cases_file), exist_ok=True)
    with open(edge_cases_file, 'w', encoding='utf-8') as f:
        for res in full_results.head(25):
            f.write(json.dumps(res, ensure_ascii=False) + '\n')

    print(f"\nДані збережено успішно!")

if __name__ == "__main__":
    main()

--- ЗАПУСК ПАЙПЛАЙНУ ЛР2 ---

--- РЕЗУЛЬТАТИ АУДИТУ (ПІСЛЯ ЛР2) ---
1. Точні дублікати: 0 (0.0%)
2. Короткі тексти (<5 слів): 13
3. Сміттєві рядки (без літер): 0
4. Середня довжина: 12.5 слів (91.4 симв.)
5. Медіанна довжина: 12 слів (87 симв.)

Дані збережено успішно!
